# generte postionnal encoding matrix PE 

In [1]:
from positional_enoding import generate_postionnal_encoding_matrix
import numpy as np
from torch.nn import Softmax
tokenised_texte = ["the" , "cat" , "is" , "so" ]

PE_matrix = generate_postionnal_encoding_matrix(tokenised_texte=tokenised_texte)
print(PE_matrix)
print(np.shape(PE_matrix))


[[0.0, 1.0, 0.0, 0.03162277660168379], [0.8414709848078965, 0.5403023058681398, 0.02660964896937897, 0.01708585911584281], [0.9092974268256817, -0.4161468365471424, 0.02875450939299445, -0.013159718445627706], [0.1411200080598672, -0.9899924966004454, 0.004462606488904997, -0.03130631155733909]]
(4, 4)


# Create the embeding Matrix Xe

In [2]:

np.random.default_rng(42)
d_model = 4
Xe = np.random.rand(d_model,d_model)
Wq = np.random.rand(d_model,d_model)
Wk = np.random.rand(d_model,d_model)
Wv = np.random.rand(d_model,d_model)
X_embeding = Xe + PE_matrix
print(X_embeding)


[[ 0.9253233   1.9828548   0.0513204   0.70306495]
 [ 1.5100946   0.84848724  0.53440634  0.85023702]
 [ 1.09760829  0.18337112  0.36487335  0.56450407]
 [ 0.43422297 -0.94536106  0.96075047  0.03086125]]


In [3]:
import torch
from torch import tensor

def attention(Xe,Wq,Wk,Wv,d_model):
    
    if Xe.shape[0] != Wq.shape[1] :
        raise Exception('dimension not correct')

    Q = np.matmul(Xe,Wq)

    if Xe.shape[0] != Wk.shape[1] :
            raise Exception('dimension not correct')

    K = np.matmul(Xe,Wk)

    if Xe.shape[0] != Wv.shape[1] :
            raise Exception('dimension not correct')

    V = np.matmul(Xe,Wv)

    Attention = np.matmul(Q,K.T)
    Attention = Attention / np.sqrt(d_model)
    Attention = tensor(Attention)
    softmax = Softmax(dim=-1)
    Attention = softmax(Attention)
    Attention = Attention.numpy()
    #Attention = np.matmul(Attention , V)

    return Attention

In [ ]:
A = attention(Xe,Wq,Wk,Wv,d_model)
print(A)


[[0.20954318 0.20987619 0.33130634 0.24927428]
 [0.21596253 0.21979329 0.2999103  0.26433387]
 [0.19860353 0.19916923 0.35037609 0.25185115]
 [0.20624184 0.20629678 0.32650924 0.26095214]]


In [7]:
WO = np.random.rand(d_model , d_model)
def layer_normalisation(X):
    epsilon = 10**-4
    mean = np.mean(X)
    std = np.std(X)
    return (X - mean) / (np.sqrt((std**2 + epsilon) ) )

def multi_head_projection(V , WO):
    Z = np.matmul(A,V)
    Y_att = np.matmul(Z,WO)
    return  Y_att

def first_risdual_addition(X_in,V,WO):
      Y_attn = multi_head_projection(V,WO)
      X_1 = X_in + Y_attn
      X_1 = layer_normalisation(d_model,d_model)
      return X_1

def postion_wise_feed_forward(X,W1,W2,bias1,bias2):
     X = np.matmul(X,W1) + bias1
     tensor_X = torch.tensor(X)
     tensor_X = torch.nn.functional.relu(tensor_X)
     X = tensor_X.numpy()
     Y_ffn = np.matmul(X,W2) + bias2
     return Y_ffn


def Encoder_output(Y_ffn):
     Y_ffn = X_embeding + Y_ffn
     H_encoder = layer_normalisation(Y_ffn)
     return H_encoder

     
     

